## Ingest Fact Data from Silver to Gold Layer

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, FloatType
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Shopvista/Shopvista-Data-Engineering-Project-_databricks/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema,gold_schema)

In [0]:
# Widget
dbutils.widgets.text('catalog', 'shopvista', 'catalog')
dbutils.widgets.text('order_source', 'order_items', 'order_source')
dbutils.widgets.text('return_source', 'order_returns', 'return_source')
dbutils.widgets.text('shipment_source', 'order_shipments', 'shipment_source')

## ORDER ITEMS

In [0]:
# set up the environment
catalog = dbutils.widgets.get("catalog")
order_source = dbutils.widgets.get("order_source")
print(catalog, order_source)

In [0]:
#Read data
df_silver = spark.table(f'{catalog}.{silver_schema}.{order_source}')
#display(df_silver.limit(5))

In [0]:
# Create a new column named gross_amount
df_gold = df_silver.withColumn('gross_amount', F.col('quantity') * F.col('unit_price'))

#display(df_gold.limit(5))

In [0]:
#  Add discount_amount (discount_pct is already numeric, e.g., 21 -> 21%)
df_gold = df_gold.withColumn('discount_amount', F.col('gross_amount') * F.col('discount_pct')/100)

#display(df_silver.limit(5))


In [0]:
# Add sale_amount = gross - discount
df_gold = df_gold.withColumn('sales_amount', F.col('gross_amount')-F.col('discount_amount')+F.col("tax_amount"))

#display(df_gold.limit(5))


In [0]:
# add date id
df_gold = df_gold.withColumn("date_id", F.date_format(F.col("dt"), "yyyyMMdd").cast(IntegerType()))  # Create date_key

#display(df_gold.limit(5))



In [0]:
# Coupon flag
#  coupon flag = 1 if coupon_code is not null else 0
df_gold = df_gold.withColumn(
    "coupon_flag",
    F.when(F.col("coupon_code").isNotNull(), F.lit(1))
     .otherwise(F.lit(0))
)

#df_gold.limit(20).display()

In [0]:
df_gold = df_gold.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("tax_amount"),
    F.col("sales_amount").alias("net_amount")
)

In [0]:
# Write to Gold table
df_gold.write\
    .format('delta')\
    .option('delta.enableChangeDatafeed', 'true')\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{gold_schema}.{order_source}')

## ORDER RETURNS

In [0]:
# set up the environment
catalog = dbutils.widgets.get("catalog")
return_source = dbutils.widgets.get("return_source")
print(catalog, return_source)


- Data Transformation & Encrichement

In [0]:
# Read the silver table
df_silver = spark.table(f'{catalog}.{silver_schema}.{return_source}')
#display(df_silver.limit(5))

In [0]:
#Add `date_id` column: yyyyMMdd format from order_dt
df_silver = df_silver.withColumn(
    "date_id",
    F.date_format(F.col("order_dt"), "yyyyMMdd")
)
#display(df_silver.limit(5))

In [0]:
#Calculate `return_days` = difference in days between return_ts and order_dt.
df_silver = df_silver.withColumn(
    "return_days",
    F.datediff(F.col("return_ts"), F.col("order_dt"))
)
#display(df_silver.limit(5))

In [0]:
#Create policy compliance flags: within_policy and is_late_return
df_silver = df_silver.withColumn(
    "within_policy",
    F.when(F.col("return_days") <= 15, 1).otherwise(0)
).withColumn(
    "is_late_return",
    F.when(F.col("return_days") > 15, 1).otherwise(0)
)
#display(df_silver.limit(5))

In [0]:
#Write to Gold table
df_silver.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', True)\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{gold_schema}.{return_source}')

## ORDER SHIPMENTS

In [0]:
catalog = dbutils.widgets.get('catalog')
shipment_source = dbutils.widgets.get('shipment_source')
print(catalog, shipment_source)

In [0]:
# Read table
df_silver = spark.table(f'{catalog}.{silver_schema}.{shipment_source}')
#display(df_silver.limit(5))

In [0]:
#Add `carrier_group`: Domestic: ECOMEXPRESS, DELHIVERY, XPRESSBEES, BLUEDART. o International: All other carriers.
df_silver = df_silver.withColumn(
    "carrier_group",
    F.when(
        F.col("carrier").isin("ECOMEXPRESS", "DELHIVERY", "XPRESSBEES", "BLUEDART"),
        "Domestic"
    ).otherwise("International")
)
#display(df_silver.limit(5))

In [0]:
#Add `is_weekend_shipment` flag: True if `order_dt` is Saturday or Sunday, else False. 
df_silver = df_silver.withColumn(
    "is_weekend_shipment",
    F.dayofweek("order_dt").isin([7, 1])
)
#display(df_silver.limit(5))

In [0]:
# Write to Gold table
df_silver.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', True)\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{gold_schema}.{shipment_source}')